Social Media-Aware Cleaning 

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType
import emoji
import re

from emoticon_fix import emoticon_fix


In [ ]:
spark = SparkSession.builder.getOrCreate()

In [ ]:
train_df = spark.read.csv('../data/twitter_training.csv')

In [ ]:
col_names = ["id", "topic", "sentiment", "content"]
train_df = train_df.toDF(*col_names)

train_df.show(5, truncate=False)

(1) Emoji Translation

In [32]:
def translate_emojis_emoticons(text):
    if text is None:
        return None
    
    text = emoticon_fix(text)
    
    text = emoji.demojize(text, delimiters=(" ", " "))
    text = text.replace("_", " ")
    text = text.replace(":", "")

    return text
    

In [33]:
emoji_emoticon_udf = udf(translate_emojis_emoticons, StringType())

In [34]:
emoji_emoticon_df = train_df.withColumn("emoji_emoticon_translated", emoji_emoticon_udf(train_df["content"]))
emoji_emoticon_df.select("content", "emoji_emoticon_translated").show(5, truncate=False)

+---------------------------------------------------------+----------------------------------------------------------+
|content                                                  |emoji_emoticon_translated                                 |
+---------------------------------------------------------+----------------------------------------------------------+
|im getting on borderlands and i will murder you all ,    |im getting on borderlands and i will murder you all ,     |
|I am coming to the borders and I will kill you all,      |I am coming to the borders and I will kill you all ,      |
|im getting on borderlands and i will kill you all,       |im getting on borderlands and i will kill you all ,       |
|im coming on borderlands and i will murder you all,      |im coming on borderlands and i will murder you all ,      |
|im getting on borderlands 2 and i will murder you me all,|im getting on borderlands 2 and i will murder you me all ,|
+-----------------------------------------------

In [35]:
# JUST SOME TEST DATA TO SEE IF THE EMOJI TRANSLATION METHOD IS WORKING lol

emoji_emoticon_test_data = [
    ("1", "test", "Positive", "I love this😂"),
    ("2", "test", "Negative", "This is terrible 😭 "),
    ("3", "test", "Neutral", "Nothing special 😐"),
    ("4", "test", "Positive", "Best day ever :)"),
    ("5", "test", "Negative", "I'm tired😩 ")
]

emoji_emoticon_test_df = spark.createDataFrame(
    emoji_emoticon_test_data,
    ["id", "topic", "sentiment", "content"]
)

In [36]:
emoji_emoticon_test_result_df = emoji_emoticon_test_df.withColumn(
    "emoji_emoticon_translated",
    emoji_emoticon_udf(emoji_emoticon_test_df["content"])
)

emoji_emoticon_test_result_df.select("content", "emoji_emoticon_translated").show(truncate=False)

+--------------------+-------------------------------------+
|content             |emoji_emoticon_translated            |
+--------------------+-------------------------------------+
|I love this😂       |I love this  face with tears of joy  |
|This is terrible 😭 |This is terrible  loudly crying face |
|Nothing special 😐  |Nothing special  neutral face        |
|Best day ever :)    |Best day ever Smile                  |
|I'm tired😩         |I ' m tired  weary face              |
+--------------------+-------------------------------------+



(2) Hashtag Splitting

In [37]:
def split_hashtags(text):
    if text is None:
        return None

    def split_tag(match):
        tag = match.group()[1:]
        tag = tag.replace("_", " ")
        tag = re.sub(r'([a-z])([A-Z])', r'\1 \2', tag)
        return tag

    return re.sub(r'#\w+', split_tag, text)

In [38]:
hashtag_udf = udf(split_hashtags, StringType())

In [39]:
hashtag_df = train_df.withColumn("hashtags_split", hashtag_udf(train_df["content"]))
hashtag_df.select("content", "hashtags_split").show(20, truncate=False)

+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|content                                                                                                                                                                                                                                                                                              |hashtags_split                                                                                                     

In [40]:
# Test data for hashtag splitting
hashtag_test_data = [
    ("1", "test", "Positive", "I love this #BestDayEver"),
    ("2", "test", "Negative", "This is terrible #WorstDay"),
    ("3", "test", "Neutral", "Just another day #MondayMood"),
    ("4", "test", "Positive", "Best day ever #BestDayEver"),
    ("5", "test", "Negative", "I hate this #WorstDay"),
    ("6", "test", "Neutral", "chilling #good_vibes"),
    ("7", "test", "Positive", "So excited #LifeIsGood"),
    ("8", "test", "Negative", "why is this happening #badluck")
]

hashtag_test_df = spark.createDataFrame(
    hashtag_test_data,
    ["id", "topic", "sentiment", "content"]
)

In [41]:
hashtag_test_result_df = hashtag_test_df.withColumn(
    "hashtags_split",
    hashtag_udf(hashtag_test_df["content"])
)

hashtag_test_result_df.select("content", "hashtags_split").show(truncate=False)

+------------------------------+-----------------------------+
|content                       |hashtags_split               |
+------------------------------+-----------------------------+
|I love this #BestDayEver      |I love this Best Day Ever    |
|This is terrible #WorstDay    |This is terrible Worst Day   |
|Just another day #MondayMood  |Just another day Monday Mood |
|Best day ever #BestDayEver    |Best day ever Best Day Ever  |
|I hate this #WorstDay         |I hate this Worst Day        |
|chilling #good_vibes          |chilling good vibes          |
|So excited #LifeIsGood        |So excited Life Is Good      |
|why is this happening #badluck|why is this happening badluck|
+------------------------------+-----------------------------+

